# Week 4 Task 1 — Fare Amount Training Dataset

## Goal

Build a reproducible trip-level machine-learning dataset from the integrated Delta table for fare amount prediction.

This notebook:

- uses `integrated_taxi_trips` as the source of truth;
- selects official NYC boroughs whose **training-period** trip count is at least 10,000;
- defines `fare_amount` as the regression label;
- derives reusable temporal and missingness features;
- excludes monetary fields that would leak the target;
- assigns chronological Train, Validation, and Test splits; and
- writes the result as a partitioned Delta table.

**Source grain:** one row per valid taxi trip.  
**Output grain:** one row per eligible taxi trip.  
**Target unit:** US dollars.

## Setup

Run the notebook from anywhere inside the repository. The setup locates the project root by finding `pyproject.toml`, so it does not depend on a user-specific absolute path.

In [1]:
from pathlib import Path
import math
import sys

from pyspark.sql import functions as F


def find_project_root(start: Path) -> Path:
    for candidate in (start, *start.parents):
        if (candidate / "pyproject.toml").is_file():
            return candidate
    raise FileNotFoundError("Could not find the project root containing pyproject.toml")


PROJECT_ROOT = find_project_root(Path.cwd().resolve())
SOURCE_PATH = PROJECT_ROOT / "data" / "processed" / "integrated_taxi_trips"
OUTPUT_PATH = PROJECT_ROOT / "data" / "processed" / "ml" / "fare_training_dataset"

# Make the local source package importable even when the notebook is opened
# without first installing the project in editable mode.
source_package = str(PROJECT_ROOT / "src")
if source_package not in sys.path:
    sys.path.insert(0, source_package)

from data_intensive_computing.spark import build_spark_session

spark = build_spark_session(
    "Week4-Task1-Fare-Training-Dataset",
    driver_memory="8g",
    shuffle_partitions=32,
)

print("Project root:", PROJECT_ROOT)
print("Integrated source:", SOURCE_PATH)
print("Training output:", OUTPUT_PATH)

D:\Master\data intensive\data-intensive-computing\.venv\Lib\site-packages\pyspark\testing\utils.py:127: FutureWarning: PySpark does not yet fully support pandas >= 3.0.0. Some features may not work correctly. It is recommended to use pandas < 3.0.0 for now.
  require_minimum_pandas_version()


Project root: D:\Master\data intensive\data-intensive-computing
Integrated source: D:\Master\data intensive\data-intensive-computing\data\processed\integrated_taxi_trips
Training output: D:\Master\data intensive\data-intensive-computing\data\processed\ml\fare_training_dataset


### Parameters and key assumptions

- Borough eligibility is calculated from the Train period only, preventing future Validation/Test records from determining the cohort.
- The expected high-volume boroughs are Bronx, Brooklyn, Manhattan, and Queens.
- Pickup and drop-off zones are assumed to be known when the fare estimate is requested.
- Recorded `trip_distance` is used as a proxy for a route-distance estimate available at prediction time.
- The raw timestamp remains in the output for audit and splitting, but it is not assembled directly into the model feature vector.
- Air-quality coverage is sparse. The notebook adds a missingness indicator; Task 2 should fit imputation using Train only.

In [2]:
# The shared raw/platform baseline currently contains January-March 2024.
# Fixed calendar boundaries keep Task 4's raw and platform workflows comparable.
TRAIN_START = "2024-01-01"
VALIDATION_START = "2024-03-01"
TEST_START = "2024-03-16"
TEST_END = "2024-04-01"

MIN_TRAIN_TRIP_COUNT = 10_000
NYC_BOROUGH_CANDIDATES = [
    "Bronx",
    "Brooklyn",
    "Manhattan",
    "Queens",
    "Staten Island",
]
EXPECTED_SELECTED_BOROUGHS = {"Bronx", "Brooklyn", "Manhattan", "Queens"}
EXPECTED_SPLITS = {"train", "validation", "test"}

print("Train:     ", TRAIN_START, "to", VALIDATION_START)
print("Validation:", VALIDATION_START, "to", TEST_START)
print("Test:      ", TEST_START, "to", TEST_END)

Train:      2024-01-01 to 2024-03-01
Validation: 2024-03-01 to 2024-03-16
Test:       2024-03-16 to 2024-04-01


## Steps

### 1. Load the integrated Delta table

The partition predicate limits the scan to 2024. The timestamp predicates below define the exact modeling period.

In [3]:
integrated = (
    spark.read.format("delta")
    .load(str(SOURCE_PATH))
    .filter(F.col("pickup_year") == 2024)
    .filter(
        (F.col("pickup_datetime") >= F.to_timestamp(F.lit(TRAIN_START)))
        & (F.col("pickup_datetime") < F.to_timestamp(F.lit(TEST_END)))
    )
)

print(f"Columns available: {len(integrated.columns)}")
integrated.select(
    "trip_id",
    "pickup_datetime",
    "pickup_borough",
    "pickup_zone",
    "dropoff_zone",
    "trip_distance",
    "fare_amount",
).show(5, truncate=False)

Columns available: 55
+----------------------------------------------------------------+-------------------+--------------+-----------------+-----------------------+-------------+-----------+
|trip_id                                                         |pickup_datetime    |pickup_borough|pickup_zone      |dropoff_zone           |trip_distance|fare_amount|
+----------------------------------------------------------------+-------------------+--------------+-----------------+-----------------------+-------------+-----------+
|00000f6b79e1d58d1802f628150ea2f605db0641eeee018d9425cbf62998f67c|2024-01-05 08:18:23|Queens        |JFK Airport      |Flushing               |10.39        |40.8       |
|000147c642efe102c2cb38d12110b309362d795e3c26e6f107ac469d3b0a6c80|2024-01-11 21:59:51|Manhattan     |East Chelsea     |Greenwich Village South|2.9          |17.7       |
|00056d66dec18c3d69c2178eae1be92c25a92d8bb29d99206335858eb0fc0390|2024-01-05 15:40:10|Manhattan     |Central Park     |Flatiron 

### 2. Select high-volume boroughs using Train only

The 10,000-trip threshold is evaluated only on records before 1 April 2024. This avoids using future data to select the modeling population.

In [4]:
train_borough_counts = (
    integrated
    .filter(F.col("pickup_datetime") < F.to_timestamp(F.lit(VALIDATION_START)))
    .filter(F.col("pickup_borough").isin(NYC_BOROUGH_CANDIDATES))
    .groupBy("pickup_borough")
    .agg(F.count("*").alias("train_trip_count"))
    .orderBy(F.desc("train_trip_count"))
)

train_borough_counts.show(truncate=False)

selected_boroughs = sorted(
    row["pickup_borough"]
    for row in train_borough_counts.collect()
    if row["train_trip_count"] >= MIN_TRAIN_TRIP_COUNT
)

assert set(selected_boroughs) == EXPECTED_SELECTED_BOROUGHS, (
    "The borough population changed. Review the threshold and source data before training. "
    f"Expected {sorted(EXPECTED_SELECTED_BOROUGHS)}, got {selected_boroughs}."
)

print("Selected boroughs:", selected_boroughs)

+--------------+----------------+
|pickup_borough|train_trip_count|
+--------------+----------------+
|Manhattan     |5293115         |
|Queens        |510939          |
|Brooklyn      |52551           |
|Bronx         |13757           |
|Staten Island |122             |
+--------------+----------------+

Selected boroughs: ['Bronx', 'Brooklyn', 'Manhattan', 'Queens']


### 3. Define the target and eligible rows

The target is the non-negative base fare in US dollars. `trip_distance` is required because it is a core route feature. Nullable predictor columns such as passenger count, weather, and PM2.5 remain nullable for Task 2 to impute using Train-only statistics.

In [5]:
eligible_trips = (
    integrated
    .filter(F.col("pickup_borough").isin(selected_boroughs))
    .filter(F.col("fare_amount").isNotNull() & (F.col("fare_amount") >= 0))
    .filter(F.col("trip_distance").isNotNull() & (F.col("trip_distance") >= 0))
)

eligible_trips.select(
    F.count("*").alias("eligible_rows"),
    F.min("pickup_datetime").alias("first_pickup"),
    F.max("pickup_datetime").alias("last_pickup"),
    F.min("fare_amount").alias("min_fare_usd"),
    F.max("fare_amount").alias("max_fare_usd"),
).show(truncate=False)

+-------------+-------------------+-------------------+------------+------------+
|eligible_rows|first_pickup       |last_pickup        |min_fare_usd|max_fare_usd|
+-------------+-------------------+-------------------+------------+------------+
|9380457      |2024-01-01 00:00:00|2024-03-31 23:59:59|0.0         |9792.0      |
+-------------+-------------------+-------------------+------------+------------+



### 4. Generate temporal and missingness features

`pickup_datetime`, `pickup_date`, and the existing timestamp-valued `pickup_hour` are not direct ML features. Instead, the pipeline extracts hour-of-day, day-of-week, month, weekend, and cyclic encodings that represent reusable temporal patterns.

In [6]:
feature_rows = (
    eligible_trips
    .withColumn("pickup_hour_of_day", F.hour("pickup_datetime"))
    .withColumn("pickup_day_of_week", F.dayofweek("pickup_datetime"))
    .withColumn("pickup_month_of_year", F.month("pickup_datetime"))
    .withColumn(
        "is_weekend",
        F.when(F.dayofweek("pickup_datetime").isin(1, 7), F.lit(1.0)).otherwise(F.lit(0.0)),
    )
    .withColumn(
        "pickup_hour_sin",
        F.sin(2 * F.lit(math.pi) * F.col("pickup_hour_of_day") / F.lit(24.0)),
    )
    .withColumn(
        "pickup_hour_cos",
        F.cos(2 * F.lit(math.pi) * F.col("pickup_hour_of_day") / F.lit(24.0)),
    )
    .withColumn(
        "pickup_dow_sin",
        F.sin(2 * F.lit(math.pi) * (F.col("pickup_day_of_week") - 1) / F.lit(7.0)),
    )
    .withColumn(
        "pickup_dow_cos",
        F.cos(2 * F.lit(math.pi) * (F.col("pickup_day_of_week") - 1) / F.lit(7.0)),
    )
    .withColumn(
        "air_quality_missing",
        F.col("air_quality_pm25_mean").isNull().cast("double"),
    )
)

### 5. Assign chronological splits

The split boundaries model future prediction and prevent trips from the same period being randomly distributed across Train and Test.

In [7]:
feature_rows = feature_rows.withColumn(
    "split",
    F.when(
        F.col("pickup_datetime") < F.to_timestamp(F.lit(VALIDATION_START)),
        F.lit("train"),
    )
    .when(
        F.col("pickup_datetime") < F.to_timestamp(F.lit(TEST_START)),
        F.lit("validation"),
    )
    .otherwise(F.lit("test")),
)

split_summary = (
    feature_rows
    .groupBy("split", "pickup_borough")
    .agg(
        F.count("*").alias("trip_count"),
        F.min("pickup_datetime").alias("first_pickup"),
        F.max("pickup_datetime").alias("last_pickup"),
        F.round(F.avg("fare_amount"), 2).alias("avg_fare_usd"),
    )
    .orderBy("split", F.desc("trip_count"))
)

actual_splits = {
    row["split"] for row in split_summary.select("split").distinct().collect()
}
assert actual_splits == EXPECTED_SPLITS, (
    "The integrated source does not cover every configured split. "
    f"Expected {sorted(EXPECTED_SPLITS)}, got {sorted(actual_splits)}. "
    "Check the source date range or update the fixed split boundaries before writing."
)

split_summary.show(20, truncate=False)

+----------+--------------+----------+-------------------+-------------------+------------+
|split     |pickup_borough|trip_count|first_pickup       |last_pickup        |avg_fare_usd|
+----------+--------------+----------+-------------------+-------------------+------------+
|test      |Manhattan     |1593854   |2024-03-16 00:00:00|2024-03-31 23:59:59|15.92       |
|test      |Queens        |160289    |2024-03-16 00:00:06|2024-03-31 23:59:56|52.4        |
|test      |Brooklyn      |23778     |2024-03-16 00:00:29|2024-03-31 23:55:36|28.27       |
|test      |Bronx         |5973      |2024-03-16 00:01:05|2024-03-31 23:58:47|30.72       |
|train     |Manhattan     |5293115   |2024-01-01 00:00:00|2024-02-29 23:59:58|15.19       |
|train     |Queens        |510939    |2024-01-01 00:00:05|2024-02-29 23:59:50|51.92       |
|train     |Brooklyn      |52551     |2024-01-01 00:02:24|2024-02-29 23:59:45|28.81       |
|train     |Bronx         |13757     |2024-01-01 00:00:03|2024-02-29 23:58:12|32

### 6. Select the leakage-safe training schema

Only the target and fields available for feature engineering are retained. `total_amount`, tips, taxes, tolls, surcharges, and other post-fare monetary components are intentionally excluded.

In [8]:
training_dataset = feature_rows.select(
    # Audit and split fields; these are not assembled into the model feature vector.
    "trip_id",
    "pickup_datetime",
    "split",
    # Regression target.
    F.col("fare_amount").alias("label"),
    # Location features.
    "pickup_zone",
    "dropoff_zone",
    "pickup_borough",
    "dropoff_borough",
    # Trip and operational features.
    "trip_distance",
    "passenger_count",
    "vendor_id",
    "rate_code_id",
    "store_and_fwd_flag",
    # Derived temporal features.
    "pickup_hour_of_day",
    "pickup_day_of_week",
    "pickup_month_of_year",
    "is_weekend",
    "pickup_hour_sin",
    "pickup_hour_cos",
    "pickup_dow_sin",
    "pickup_dow_cos",
    # Weather features.
    "weather_temperature_c",
    "weather_relative_humidity_pct",
    "weather_precipitation_mm",
    "weather_snow_depth_mm",
    "weather_wind_speed_kmh",
    "weather_pressure_hpa",
    "weather_condition_code",
    # Air-quality value and explicit missingness.
    "air_quality_pm25_mean",
    "air_quality_missing",
)

training_dataset.printSchema()

root
 |-- trip_id: string (nullable = true)
 |-- pickup_datetime: timestamp (nullable = true)
 |-- split: string (nullable = false)
 |-- label: double (nullable = true)
 |-- pickup_zone: string (nullable = true)
 |-- dropoff_zone: string (nullable = true)
 |-- pickup_borough: string (nullable = true)
 |-- dropoff_borough: string (nullable = true)
 |-- trip_distance: double (nullable = true)
 |-- passenger_count: integer (nullable = true)
 |-- vendor_id: long (nullable = true)
 |-- rate_code_id: long (nullable = true)
 |-- store_and_fwd_flag: string (nullable = true)
 |-- pickup_hour_of_day: integer (nullable = true)
 |-- pickup_day_of_week: integer (nullable = true)
 |-- pickup_month_of_year: integer (nullable = true)
 |-- is_weekend: double (nullable = false)
 |-- pickup_hour_sin: double (nullable = true)
 |-- pickup_hour_cos: double (nullable = true)
 |-- pickup_dow_sin: double (nullable = true)
 |-- pickup_dow_cos: double (nullable = true)
 |-- weather_temperature_c: double (nullabl

### 7. Write the training Delta table

The full Task 1 dataset is rebuilt deterministically and partitioned by split. Task 2 can load one split at a time and fit imputers and categorical encoders on Train only.

In [10]:
(
    training_dataset.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .partitionBy("split")
    .save(str(OUTPUT_PATH))
)

print("Fare training Delta table written to:", OUTPUT_PATH)

Fare training Delta table written to: D:\Master\data intensive\data-intensive-computing\data\processed\ml\fare_training_dataset


## Checks

Reload the persisted dataset and verify the target, population, split labels, and leakage-sensitive schema.

In [11]:
persisted = spark.read.format("delta").load(str(OUTPUT_PATH))

quality = persisted.agg(
    F.count("*").alias("row_count"),
    F.sum(F.col("label").isNull().cast("long")).alias("null_labels"),
    F.sum((F.col("label") < 0).cast("long")).alias("negative_labels"),
    F.countDistinct("trip_id").alias("distinct_trip_ids"),
).first()

assert quality["row_count"] > 0
assert quality["null_labels"] == 0
assert quality["negative_labels"] == 0
assert quality["distinct_trip_ids"] == quality["row_count"], "Duplicate trip_id detected"

assert set(
    row["split"] for row in persisted.select("split").distinct().collect()
) == EXPECTED_SPLITS
assert set(
    row["pickup_borough"]
    for row in persisted.select("pickup_borough").distinct().collect()
) == EXPECTED_SELECTED_BOROUGHS

for forbidden in {
    "tip_amount",
    "total_amount",
    "extra",
    "mta_tax",
    "tolls_amount",
    "improvement_surcharge",
    "congestion_surcharge",
    "airport_fee",
    "pickup_date",
    "pickup_hour",
}:
    assert forbidden not in persisted.columns, f"Leakage or redundant column found: {forbidden}"

print(dict(quality.asDict()))
(
    persisted
    .groupBy("split")
    .agg(
        F.count("*").alias("row_count"),
        F.round(F.avg("label"), 2).alias("avg_fare_usd"),
        F.round(F.min("label"), 2).alias("min_fare_usd"),
        F.round(F.max("label"), 2).alias("max_fare_usd"),
    )
    .orderBy("split")
    .show(truncate=False)
)

{'row_count': 9380457, 'null_labels': 0, 'negative_labels': 0, 'distinct_trip_ids': 9380457}
+----------+---------+------------+------------+------------+
|split     |row_count|avg_fare_usd|min_fare_usd|max_fare_usd|
+----------+---------+------------+------------+------------+
|test      |1783894  |19.41       |0.0         |800.0       |
|train     |5870362  |18.55       |0.0         |9792.0      |
|validation|1726201  |19.02       |0.0         |900.0       |
+----------+---------+------------+------------+------------+

